# SemEval-2026 Task 13 — Subtask B (Google Colab Version)

This notebook is adapted from the Kaggle starter to run on **Google Colab**.

**Key differences from the Kaggle version:**
- Data is downloaded from HuggingFace instead of using Kaggle input paths
- Fixed deprecated `transformers` API calls (`eval_strategy`, `processing_class`)

Make sure you select a **GPU runtime** in Colab:
`Runtime → Change runtime type → T4 GPU`

In [ ]:
!pip install --upgrade transformers==4.45.0 huggingface_hub datasets scikit-learn

In [ ]:
# Download the dataset from HuggingFace
from datasets import load_dataset

print("Downloading SemEval-2026-Task13 Subtask B dataset from HuggingFace...")
hf_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "B")
print("Dataset downloaded successfully!")
print(hf_dataset)

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import warnings
warnings.filterwarnings("ignore")

We suggest using a single class, it will make refinement easier.

In your implementation, feel free to update the training procedure, change model and do whatever feels right.

In [ ]:
class CodeBERTTrainer:
    def __init__(self, max_length=512, model_name="microsoft/codebert-base"):
        self.max_length = max_length
        self.model_name = model_name
        self.tokenizer = None
        self.model = None
        self.num_labels = None

    def load_and_prepare_data(self):
        try:
            # Use the HuggingFace dataset that was already downloaded
            df = hf_dataset['train'].to_pandas()

            print(f"Dataset columns: {df.columns.tolist()}")
            print(f"Sample data:\n{df.head()}")

            if 'code' not in df.columns or 'label' not in df.columns:
                raise ValueError("Dataset must contain 'code' and 'label' columns")

            df = df.dropna(subset=['code', 'label'])
            df['label'] = df['label'].astype(int)
            self.num_labels = df['label'].nunique()

            print(f"Number of unique labels: {self.num_labels}")
            print(f"Label range: {df['label'].min()} to {df['label'].max()}")
            print(f"Label distribution:\n{df['label'].value_counts().sort_index()}")

            # Use the validation split from HuggingFace
            val_df = hf_dataset['validation'].to_pandas()

            print(f"Train samples: {len(df)}, Validation samples: {len(val_df)}")

            return df, val_df

        except Exception as e:
            print(f"Error loading dataset: {e}")
            raise

    def initialize_model_and_tokenizer(self):
        print(f"Initializing {self.model_name} model and tokenizer...")

        self.tokenizer = RobertaTokenizer.from_pretrained(self.model_name)

        self.model = RobertaForSequenceClassification.from_pretrained(
            self.model_name,
            num_labels=self.num_labels,
            problem_type="single_label_classification"
        ).to('cuda')

        print(f"Model initialized with {self.num_labels} labels")

    def tokenize_function(self, examples):
        return self.tokenizer(
            examples['code'],
            truncation=True,
            padding=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

    def prepare_datasets(self, train_df, val_df):
        print("Preparing datasets for training...")

        train_dataset = Dataset.from_pandas(train_df[['code', 'label']])
        val_dataset = Dataset.from_pandas(val_df[['code', 'label']])

        train_dataset = train_dataset.map(
            self.tokenize_function,
            batched=True,
            remove_columns=['code']
        )
        val_dataset = val_dataset.map(
            self.tokenize_function,
            batched=True,
            remove_columns=['code']
        )

        train_dataset = train_dataset.rename_column('label', 'labels')
        val_dataset = val_dataset.rename_column('label', 'labels')

        return train_dataset, val_dataset

    def compute_metrics(self, eval_pred):
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)

        accuracy = accuracy_score(labels, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')

        return {
            'accuracy': accuracy,
            'f1': f1,
            'precision': precision,
            'recall': recall
        }

    def train(self, train_dataset, val_dataset, output_dir="./results", num_epochs=3, batch_size=16, learning_rate=2e-5):
        print("Starting training...")
        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=num_epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            weight_decay=0.01,
            logging_steps=5,
            eval_strategy="steps",
            eval_steps=500,
            save_strategy="steps",
            save_steps=500,
            load_best_model_at_end=True,
            metric_for_best_model="f1",
            greater_is_better=True,
            remove_unused_columns=False,
            learning_rate=learning_rate,
            lr_scheduler_type="linear",
            save_total_limit=2,
            report_to="none",
        )

        data_collator = DataCollatorWithPadding(tokenizer=self.tokenizer)

        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            processing_class=self.tokenizer,
            data_collator=data_collator,
            compute_metrics=self.compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
        )
        print("Start training")
        trainer.train()

        trainer.save_model()
        self.tokenizer.save_pretrained(output_dir)

        print(f"Training completed. Model saved to {output_dir}")

        return trainer

    def evaluate_model(self, trainer, val_dataset):
        print("Evaluating model...")

        predictions = trainer.predict(val_dataset)
        y_pred = np.argmax(predictions.predictions, axis=1)
        y_true = predictions.label_ids

        print("Classification Report:")
        print(classification_report(y_true, y_pred))

        return predictions

    def run_full_pipeline(self, output_dir="./results", num_epochs=3, batch_size=16, learning_rate=2e-5):
        try:
            train_df, val_df = self.load_and_prepare_data()

            self.initialize_model_and_tokenizer()

            train_dataset, val_dataset = self.prepare_datasets(train_df, val_df)

            trainer = self.train(
                train_dataset, val_dataset,
                output_dir=output_dir,
                num_epochs=num_epochs,
                batch_size=batch_size,
                learning_rate=learning_rate
            )

            self.evaluate_model(trainer, val_dataset)

            print("Pipeline completed successfully!")
            return trainer

        except Exception as e:
            print(f"Error in pipeline: {e}")
            raise

In [ ]:
OUTPUT_DIR = "taskB-model"

trainer_obj = CodeBERTTrainer(
    max_length=256,
)

t = trainer_obj.run_full_pipeline(
    output_dir=OUTPUT_DIR,
    num_epochs=10,
    batch_size=16,
    learning_rate=2e-5
)

In [ ]:
import torch
import logging
from itertools import chain
from datasets import load_dataset
from tqdm import tqdm


@torch.no_grad()
def predict_with_trainer(trainer_obj, parquet_path, output_path, max_length=512, batch_size=16, device=None):
    """
    Uses trainer_obj.model and trainer_obj.tokenizer to run streaming inference
    over a parquet file with columns ['ID','code'] and writes 'ID,prediction' CSV.
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model = trainer_obj.model
    tokenizer = trainer_obj.tokenizer if hasattr(trainer_obj, "tokenizer") else None
    if tokenizer is None:
        raise ValueError("trainer_obj must have a tokenizer.")

    model.to(device)
    model.eval()

    ds = load_dataset("parquet", data_files=parquet_path, split="train", streaming=True)

    it = iter(ds)
    first = next(it)
    if not {"ID", "code"}.issubset(first.keys()):
        raise ValueError("Parquet file must contain 'ID' and 'code' columns")
    stream = chain([first], it)

    def batcher(iterator, bs):
        buf = []
        for ex in iterator:
            buf.append(ex)
            if len(buf) == bs:
                yield buf
                buf = []
        if buf:
            yield buf

    with open(output_path, "w") as f:
        f.write("ID,prediction\n")

        for batch in tqdm(batcher(stream, batch_size), desc="Predicting"):
            codes = [row["code"] for row in batch]
            ids   = [row["ID"] for row in batch]

            enc = tokenizer(
                codes,
                truncation=True,
                padding=True,
                max_length=max_length,
                return_tensors="pt",
            )
            input_ids = enc["input_ids"].to(device)
            attention_mask = enc["attention_mask"].to(device)

            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            pred_labels = logits.argmax(dim=-1).cpu().tolist()

            for ex_id, pred in zip(ids, pred_labels):
                f.write(f"{ex_id},{pred}\n")

    print(f"Predictions saved to {output_path}")

### Prediction on Test Set

**Note:** The test parquet file needs to be available. You can either:
1. Download it from HuggingFace (if available in the dataset)
2. Upload it manually to Colab
3. Mount Google Drive and place it there

Update the `TEST_PARQUET` path below accordingly.

In [ ]:
# Option 1: If test split is available in the HuggingFace dataset
# Save it locally so the predict function can read it
if 'test' in hf_dataset:
    hf_dataset['test'].to_parquet('test.parquet')
    TEST_PARQUET = 'test.parquet'
    print("Test parquet saved from HuggingFace dataset.")
else:
    # Option 2: Upload manually or specify path
    # from google.colab import files
    # uploaded = files.upload()  # Upload your test.parquet file here
    TEST_PARQUET = 'test.parquet'  # Update this path as needed
    print("No test split found in HuggingFace dataset. Please upload test.parquet manually.")

OUT_CSV = "submission.csv"

predict_with_trainer(
    trainer_obj=t,
    parquet_path=TEST_PARQUET,
    output_path=OUT_CSV,
    max_length=256,
    batch_size=32,
    device="cuda"
)

print("Wrote:", OUT_CSV)

In [ ]:
# Download the submission CSV to your local machine
from google.colab import files
files.download(OUT_CSV)